# Computational cost of the multi-fidelity surrogate — measured on GPU

Runs the **actual production Arm-3 scripts** (the ones in `release_webapp/`, copied into
`scripts/`), then reads six indices out of the namespace each one leaves behind. Nothing here
is a reimplementation, so `Training (s)` is a measurement rather than a CPU→GPU extrapolation.

| index | definition |
|---|---|
| Parameters | summed over the four live modules: FiLM + representation + head + load skip |
| Model (MB) | Parameters × 4 / 1024² (fp32) |
| Training (s) | the script's own wall clock |
| Inference (s) | one sample's full free-running overlap-add rollout, batch 1, median of 3 |
| Activation (MB) | summed intermediate activation bytes, batch 1, one window, forward hooks |
| Peak GPU (MB) | `torch.cuda.max_memory_allocated`, reported separately for train and rollout |
| Database (MB) | size on disk of the `.npz` the script loads |

**Configurations are the published ones**, not defaults: the tunnel runs uncapped (N = 1000
→ 700/150/150) and the frame uses the CPU-affordable override from
`run_frame3_seedsweep.py` (L = 150, STEP = 150, HIDDEN = 128, LAYERS = 1, EPOCHS = 10) —
which is what produced every reported frame number, not the notebook's HIDDEN = 192 defaults.

**Requires a GPU runtime.** Runtime → Change runtime type → T4.

Expect roughly **8–20 min total**. Each example writes its own JSON and is skipped on
re-entry, so a disconnect costs at most one example.


## 1 · GPU check

Stop if this says CPU — the numbers would then be CPU numbers.


In [ ]:
import platform, torch
print("torch :", torch.__version__, "| python", platform.python_version())
print("cuda  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu   :", torch.cuda.get_device_name(0),
          "| %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("\n!! No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")


## 2 · Point at the bundle

Upload `colab_cost/` to Drive with `data/` and `scripts/` intact, then fix `BASE` if you put
it somewhere other than `MyDrive/colab_cost`.


In [ ]:
import os, shutil, time

from google.colab import drive
drive.mount("/content/drive")

# ── edit this one line if you put the folder somewhere else ──
BASE = "/content/drive/MyDrive/colab_cost"

# Work on the local disk, not the Drive mount: these scripts write checkpoints and PNGs
# beside their database, and Drive is a network filesystem.
LOCAL = "/content/cost"
os.makedirs(LOCAL, exist_ok=True)
t0 = time.perf_counter()
for sub in ("data", "scripts"):
    src, dst = os.path.join(BASE, sub), os.path.join(LOCAL, sub)
    os.makedirs(dst, exist_ok=True)
    assert os.path.isdir(src), f"missing {src}"
    for f in sorted(os.listdir(src)):
        s, d = os.path.join(src, f), os.path.join(dst, f)
        if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
            shutil.copy2(s, d)
        print(f"  {sub}/{f:44s} {os.path.getsize(d)/1e6:7.1f} MB")
print(f"staged in {time.perf_counter()-t0:.0f}s")

OUTD = os.path.join(BASE, "results")          # results go to Drive so they survive
os.makedirs(OUTD, exist_ok=True)
print("results ->", OUTD)


## 3 · The measurement driver

Written to a file so each example runs in its own **subprocess**: these scripts are 700-line
top-level programs that allocate the whole windowed dataset, and a fresh process per example
keeps one example's memory from affecting the next one's timings.


In [ ]:
%%writefile /content/cost/measure_one.py
"""Measure the six cost indices for ONE example, on whatever device torch selects.

Runs the actual production Arm-3 script, then reads the indices out of the namespace it
leaves behind:

  Parameters   summed over the four live modules (film + representation + head + load skip)
  Model (MB)   Parameters x 4 / 1024**2   (fp32)
  Training     the script's own wall clock -- `wall` for the wt lineage, else parsed from
               the line it prints
  Inference    one sample's full free-running overlap-add rollout, batch 1, median of 3
  Activation   summed intermediate activation bytes, batch 1, one window, via forward hooks
               on every leaf module  (plus peak CUDA allocation, which is what a GPU user
               actually budgets for)
  Database     size on disk of the .npz the script loads

Usage:  python3 measure_one.py <sdof|frame3|tunnel|turbine> --out results
"""
import argparse
import contextlib
import io
import json
import os
import re
import shutil
import statistics as st
import sys
import time

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
DATA = os.path.join(HERE, "data")
SCRIPTS = os.path.join(HERE, "scripts")

SPEC = {
    "sdof": dict(kind="sdof", script="sdof_arm3_noepp_db.py",
                 db="sdof_eta60_n1500.npz",
                 env={"DB": "sdof_eta60_n1500.npz"}),
    # N_MAX=0 -> no cap -> N=1000, i.e. the 700/150/150 run the system table quotes
    "tunnel": dict(kind="wt", script="red_to_nl_tunnel_half.py",
                   db="tunnel_moving_train_database.npz",
                   env={"N_MAX": "0", "SEED": "42", "TAG": "cost_tunnel"}),
    "turbine": dict(kind="wt", script="wt_arm3_v20.py", db="wt_v20_full.npz",
                    env={"DB_PATH": "wt_v20_full.npz", "SEED": "42", "TAG": "cost_turbine",
                         "N_MAX": "0", "FLIP": "1", "STEP": "50", "SAMPLE_W": "0"}),
    # the frame's published config is the notebook's cells with the override from
    # run_frame3_seedsweep.py: L=150 STEP=150 HIDDEN=128 LAYERS=1 EPOCHS=10
    "frame3": dict(kind="frame", script="frame3_v3_colab.ipynb",
                   db="frame3_v3_database.npz", env={}),
}


class Tee(io.TextIOBase):
    """Show the script's output live and keep a copy to parse the training time out of."""

    def __init__(self, real):
        self.real, self.buf = real, io.StringIO()

    def write(self, s):
        self.real.write(s); self.real.flush(); self.buf.write(s); return len(s)

    def flush(self):
        self.real.flush()


def stage(name):
    """work/<name>/ with the database and the script in place; chdir into it."""
    sp = SPEC[name]
    work = os.path.join(HERE, "work", name)
    os.makedirs(work, exist_ok=True)
    for f in (sp["db"],):
        dst = os.path.join(work, f)
        if not os.path.exists(dst):
            try:
                os.link(os.path.join(DATA, f), dst)
            except OSError:
                shutil.copy2(os.path.join(DATA, f), dst)
    src = os.path.join(SCRIPTS, sp["script"])
    shutil.copy2(src, os.path.join(work, sp["script"]))
    os.chdir(work)
    for k, v in sp["env"].items():
        os.environ[k] = v
    os.environ["MPLBACKEND"] = "Agg"
    return work, sp


def activation_mb(mods, call):
    """Summed intermediate activation bytes for one batch-1 forward."""
    tot = [0]

    def hook(_m, _i, out):
        for t in (out if isinstance(out, (tuple, list)) else (out,)):
            if torch.is_tensor(t):
                tot[0] += t.numel() * t.element_size()

    hs = [m.register_forward_hook(hook) for mod in mods
          for m in mod.modules() if not list(m.children())]
    with torch.no_grad():
        call()
    for h in hs:
        h.remove()
    return tot[0] / 1024 ** 2


def timed_infer(fn, ids, reps=3):
    fn(ids)                                                   # warm up
    ts = []
    for _ in range(reps):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t = time.perf_counter(); fn(ids)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        ts.append(time.perf_counter() - t)
    return st.median(ts)


def parse_train_s(text, patterns):
    for p in patterns:
        m = re.search(p, text)
        if m:
            return float(m.group(1))
    return float("nan")


def run(name):
    work, sp = stage(name)
    tee = Tee(sys.stdout)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_all = time.perf_counter()

    if sp["kind"] == "frame":
        ns, mods, infer, meta = run_frame(sp, tee)
    else:
        ns, mods, infer, meta = run_script(sp, tee)

    peak_train = (torch.cuda.max_memory_allocated() / 1024 ** 2
                  if torch.cuda.is_available() else float("nan"))
    npar = sum(p.numel() for m in mods for p in m.parameters())
    L_all, nch, nst = meta["L_all"], meta["n_time_ch"], meta["n_static"]

    for m in mods:
        m.eval()
    x = torch.zeros(1, L_all, nch, device=meta["device"])
    s = torch.zeros(1, 1, nst, device=meta["device"])
    act = activation_mb(mods[:3], lambda: meta["one_window"](x, s))

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    inf = timed_infer(infer, np.array([meta["sid"]]))
    peak_infer = (torch.cuda.max_memory_allocated() / 1024 ** 2
                  if torch.cuda.is_available() else float("nan"))

    return dict(
        example=name, parameters=npar, model_mb=npar * 4 / 1024 ** 2,
        training_s=meta["train_s"], inference_s=inf, activation_mb=act,
        peak_gpu_train_mb=peak_train, peak_gpu_infer_mb=peak_infer,
        database_mb=os.path.getsize(os.path.join(work, sp["db"])) / 1e6,
        n_train=meta.get("n_train"), n_test=meta.get("n_test"),
        L_all=L_all, n_time_ch=nch, n_static=nst, sid=meta["sid"],
        device=str(meta["device"]),
        gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
        torch=torch.__version__, total_wall_s=time.perf_counter() - t_all)


def run_script(sp, tee):
    """sdof and the wt lineage: exec the script whole, then read its namespace."""
    src = open(sp["script"]).read()
    ns = {"__name__": "__main__", "__file__": sp["script"]}
    with contextlib.redirect_stdout(tee):
        exec(compile(src, sp["script"], "exec"), ns)
    out = tee.buf.getvalue()

    if sp["kind"] == "wt":
        film, rep, head, lskip = ns["film"], ns["rep"], ns["head"], ns["lskip"]
        train_s = float(ns.get("wall", parse_train_s(out, [r"train_time\s*=\s*([\d.]+)"])))
        infer = lambda ids: ns["infer"](film, rep, head, lskip, ids)
    else:                                                     # sdof
        film, rep, head, lskip = (ns["film_agg"], ns["representation"],
                                  ns["pred_head"], ns["force_skip"])
        train_s = parse_train_s(out, [r"Training\s*\|\s*wall:\s*([\d.]+)s"])
        infer = lambda ids: ns["infer_overlap_add"](ids)
    mods = [film, rep, head, lskip]

    def one_window(x, s):
        z = rep(film(x, s))
        return head(z[0] if isinstance(z, (tuple, list)) else z)

    te = np.asarray(ns["idx_te"]).ravel().astype(int)
    meta = dict(train_s=train_s, L_all=ns["L_in"] + ns["L_out"],
                n_time_ch=ns["N_TIME_CH"], n_static=ns["N_STATIC"],
                device=ns["device"], one_window=one_window, sid=int(te[0]),
                n_train=len(np.asarray(ns["idx_tr"]).ravel()), n_test=len(te))
    return ns, mods, infer, meta


def run_frame(sp, tee):
    """The frame's published run: the notebook's cells with the CPU-affordable override
    from run_frame3_seedsweep.py -- but left on the GPU, which is the whole point here."""
    code = [''.join(c['source']) for c in json.load(open(sp["script"]))['cells']
            if c['cell_type'] == 'code']
    G = {}
    # cell 0 = imports + CUDA device + seed.  Cell 1 is SKIPPED: it either mounts Drive or
    # shells out to gen_3story_frame_v3.py to REGENERATE the database.  Setting DB by hand
    # and jumping to cell 2 is exactly what run_frame3_seedsweep.py does.
    src0 = re.sub(r"^SEED = .*$", "SEED = 42", code[0], count=1, flags=re.M)
    with contextlib.redirect_stdout(tee):
        exec(src0, G)
        G["DB"] = sp["db"]
        for c in (code[2], code[3], code[4], code[5]):
            exec(c, G)
        src5 = code[5].replace("L_IN = L_OUT = 200", "L_IN = L_OUT = 150")
        for k, v in dict(STEP=150, HIDDEN=128, LAYERS=1, BATCH=64, EPOCHS=10).items():
            src5 = re.sub(rf"^{k}(\s*)= .*$", f"{k}\\1= {v}", src5, count=1, flags=re.M)
        exec(src5, G)
        assert G["L_ALL"] == G["L_IN"] + G["L_OUT"], (G["L_IN"], G["L_OUT"], G["L_ALL"])
        print(f"config: L_IN={G['L_IN']} STEP={G['STEP']} HIDDEN={G['HIDDEN']} "
              f"LAYERS={G['LAYERS']} EPOCHS={G['EPOCHS']}")
        exec(code[6], G)        # channels
        exec(code[7], G)        # tukey / rollout / run_arm -- definitions only
        # cells 8-11 are NOT run: cell 8 would call run_arm(3) itself and cell 9 trains Arm 2
        mods = list(G["run_arm"](3, "cost"))
    train_s = parse_train_s(tee.buf.getvalue(), [r"best val_mse=[\d.]+\s+train\s+([\d.]+)s"])
    film, rep, head, skip = mods
    S = G["u_nl"].shape[2]                        # (N, T, S) after cell 2's transpose

    def one_window(x, s):
        return head(rep(film(x, s))) + skip(x[:, -G["L_OUT"]:, :S], s.squeeze(1))

    te = np.asarray(G["idx_te"]).ravel().astype(int)
    infer = lambda ids: G["rollout"](mods, ids, 3)
    meta = dict(train_s=train_s, L_all=G["L_ALL"], n_time_ch=4 * S + 2,
                n_static=G["N_STATIC"],
                device=G["device"], one_window=one_window, sid=int(te[0]),
                n_train=len(np.asarray(G["idx_tr"]).ravel()), n_test=len(te))
    return G, mods, infer, meta


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("example", choices=list(SPEC))
    ap.add_argument("--out", default="results")
    a = ap.parse_args()
    outdir = a.out if os.path.isabs(a.out) else os.path.join(HERE, a.out)
    os.makedirs(outdir, exist_ok=True)
    r = run(a.example)
    p = os.path.join(outdir, f"{a.example}.json")
    with open(p + ".tmp", "w") as f:
        json.dump(r, f, indent=1); f.flush(); os.fsync(f.fileno())
    os.replace(p + ".tmp", p)
    print(f"\n=== {a.example}: params={r['parameters']:,}  train={r['training_s']:.0f}s  "
          f"infer={r['inference_s']:.3f}s  act={r['activation_mb']:.2f}MB  "
          f"peakGPU={r['peak_gpu_train_mb']:.0f}MB")
    print(f"wrote {p}")


## 4 · Measure

One subprocess per example, resumable. Order is cheapest-first so a failure surfaces early.


In [ ]:
import json, os, subprocess, sys, time

EXAMPLES = ["frame3", "sdof", "turbine", "tunnel"]     # cheapest first

def done(e):
    p = os.path.join(OUTD, f"{e}.json")
    if not os.path.exists(p):
        return False
    try:
        return "training_s" in json.load(open(p))
    except Exception:
        print(f"  corrupt {e}.json -> will re-run"); return False

t0 = time.perf_counter()
for e in EXAMPLES:
    if done(e):
        print(f"=== {e:8s} SKIP (done)"); continue
    print(f"\n{'='*70}\n=== {e}\n{'='*70}", flush=True)
    r = subprocess.run([sys.executable, "-u", os.path.join(LOCAL, "measure_one.py"),
                        e, "--out", OUTD], cwd=LOCAL)
    if r.returncode:
        print(f"!! {e} failed (rc={r.returncode}) -- continuing")
    print(f"--- elapsed {(time.perf_counter()-t0)/60:.0f} min", flush=True)

n = sum(done(e) for e in EXAMPLES)
print(f"\n{n}/{len(EXAMPLES)} measured in {(time.perf_counter()-t0)/60:.0f} min")
if n < len(EXAMPLES):
    print("Re-run cells 1-3 then this cell to continue.")


## 5 · The table


In [ ]:
import csv, glob, json, os

NAME = {"sdof": "Single-degree-of-freedom", "frame3": "Three-storey steel frame",
        "tunnel": "Buried tunnel", "turbine": "Wind turbine"}
ORDER = ["sdof", "frame3", "tunnel", "turbine"]
R = {}
for f in glob.glob(os.path.join(OUTD, "*.json")):
    r = json.load(open(f)); R[r["example"]] = r
rows = [R[e] for e in ORDER if e in R]

hdr = ["System", "Parameters", "Model (MB)", "Training (s)", "Inference (s)",
       "Activation (MB)", "Peak GPU (MB)", "Database (MB)"]
print(f"{hdr[0]:26s}{hdr[1]:>12s}{hdr[2]:>11s}{hdr[3]:>13s}{hdr[4]:>13s}"
      f"{hdr[5]:>16s}{hdr[6]:>14s}{hdr[7]:>13s}")
for r in rows:
    print(f"{NAME[r['example']]:26s}{r['parameters']:>12,}{r['model_mb']:>11.2f}"
          f"{r['training_s']:>13.0f}{r['inference_s']:>13.3f}{r['activation_mb']:>16.2f}"
          f"{r['peak_gpu_train_mb']:>14.0f}{r['database_mb']:>13.0f}")
print()
for r in rows:
    print(f"  {r['example']:8s} n_train={r['n_train']:5d} n_test={r['n_test']:4d} "
          f"L_all={r['L_all']:4d} channels={r['n_time_ch']:3d} statics={r['n_static']:3d} "
          f"| {r['device']} {r['gpu']}")

p = os.path.join(OUTD, "cost_table_gpu.csv")
with open(p, "w", newline="") as fh:
    w = csv.DictWriter(fh, list(rows[0])); w.writeheader(); w.writerows(rows)
print(f"\n-> {p}")


---

## Reading these numbers

**Training is now measured, so drop the 21.7× extrapolation.** That factor came from the
`HID = 96` backbone harness, not from these ~0.24–1.10 M-parameter production models; compare
the two once you have this table, since the gap tells you how well the bigger GRUs use the
GPU.

**Inference stays a batch-1 number** — the cost of predicting one new record, which is the
operationally meaningful figure and keeps the free-running rollout's sequential dependency
intact. It does not speed up on a GPU nearly as much as training does, because the
overlap-add bookkeeping between hops is NumPy on the host. Batching 32 records at once is
roughly 4× cheaper per record.

**Activation vs Peak GPU are different things and both are reported.** Activation is the
summed intermediate bytes for a single batch-1 window, comparable to the CPU table. Peak GPU
is `max_memory_allocated` across real training, so it includes the full batch, gradients,
and the AdamW state — that is the number to quote if a reader asks what hardware they need.

**Two configurations are non-default on purpose.** The tunnel runs uncapped (N = 1000,
700/150/150) to match the system table; set `N_MAX=500` in `measure_one.py` if you quote the
5-seed N = 500 row instead. The frame uses the seed-sweep override, which is a deliberately
reduced budget — it is cheap in this table because it was *trained* cheaply, not because the
example is easier. If you want the notebook's own HIDDEN = 192 / LAYERS = 2 / EPOCHS = 40
configuration instead, that is the 1,115,943-parameter model, and no reported accuracy number
comes from it.

**Seed 42 only.** Training time varies little between seeds (SDOF was 2335 ± 242 s on CPU
over 5 seeds, ~10%), so a single seed is adequate for a cost table in a way it is not for an
accuracy table.
